In [1]:
import os
import cv2
import torch
import glob
import gc 

import time
import torch.amp
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchmetrics.detection.mean_ap import MeanAveragePrecision

from ultralytics import YOLO, RTDETR
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F

import albumentations as A
from albumentations.pytorch import ToTensorV2

import csv
import types

In [ ]:
FOLDS_DIR = 'dataset_folds'
NUM_FOLDS = 5
EPOCHS_YOLO = 70
EPOCHS_DETR = 100

BATCH_SIZE = 8
NUM_CLASSES = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
def free_memory():
    """Агрессивная очистка RAM и VRAM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def on_epoch_end_callback(*args, **kwargs):
    """Коллбек для Ultralytics, вызывается в конце каждой эпохи."""
    free_memory()

In [8]:
def train_yolo11_model(fold_idx, fold_dir):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))
    
    print(f"--- Обучение YOLO11m на Фолде {fold_idx} ---")
    yolo_model = YOLO('yolo11m.pt')
    
    # Коллбек очистки памяти после каждой эпохи
    yolo_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)
    
    yolo_model.train(
        data=yaml_path,
        epochs=EPOCHS_YOLO,
        batch=BATCH_SIZE,
        imgsz=640,
        patience=15,                    # Early Stopping
        project='runs/ensemble_training',
        name=f'yolo11m_fold_{fold_idx}',
        device=0,
        workers=4,
        augment=True,
        
        box=7.5,                        
        cls=1.0,
        dfl=1.5,
        cos_lr=True,    

        fliplr=0.5,         # Отзеркаливание по горизонтали 
        flipud=0.0,         # Отзеркаливание по вертикали 
        degrees=30.0,        # Вращение 
        translate=0.2,      # Сдвиг 
        scale=0.6,          # Масштаб (+-60%)
        shear=20.0,          # Сдвиг/Перекос 
        perspective=0.0001,
        
        hsv_h=0.015,        # Сдвиг оттенка. 
        hsv_s=0.3,          # Насыщенность (70%). 
        hsv_v=0.3,          # Яркость (50%). 
        bgr=0.0,            # Переворот каналов BGR->RGB 

        close_mosaic=10,
        mosaic=0.5,
        mixup=0.15,
        copy_paste=0.3,
        erasing=0.4,
       
        multi_scale=0.25
    )
    
    # Глобальная очистка
    del yolo_model
    free_memory()

In [ ]:
def custom_optimizer_step(self):
    """Кастомный шаг оптимизатора с жестким клиппингом градиентов для RT-DETR"""
    self.scaler.unscale_(self.optimizer)
    
    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
    
    self.scaler.step(self.optimizer)
    self.scaler.update()
    self.optimizer.zero_grad()
    
    if self.ema:
        self.ema.update(self.model)

def on_train_start_add_clip(trainer):
    trainer.optimizer_step = types.MethodType(custom_optimizer_step, trainer)
    print("ВНИМАНИЕ: Активирован кастомный Gradient Clipping (max_norm=0.1)!")

def train_rtdetr_model(fold_idx, fold_dir):
    yaml_path = os.path.abspath(os.path.join(fold_dir, 'data.yaml'))

    print(f"--- Обучение RT-DETR на Фолде {fold_idx} ---")
    rtdetr_model = RTDETR('rtdetr-l.pt')
    
    rtdetr_model.add_callback("on_fit_epoch_end", on_epoch_end_callback)
    rtdetr_model.add_callback("on_train_start", on_train_start_add_clip)

    rtdetr_model.train(
        data=yaml_path,
        epochs=EPOCHS_DETR,
        batch=BATCH_SIZE,
        imgsz=640,
        patience=15,                    # Early Stopping
        project='runs/ensemble_training',
        name=f'rtdetr_fold_{fold_idx}',
        device=0,
        workers=4,
        augment=True,
        
        amp=True,

        save_period=10,
        pretrained=True,

        box=5.0,                        
        cls=1.0,
        dfl=1.5,

        optimizer='AdamW',  
        lr0=0.0001,
        cos_lr=True,

        fliplr=0.5,         # Отзеркаливание по горизонтали 
        flipud=0.0,         # Отзеркаливание по вертикали 
        degrees=30.0,        # Вращение 
        translate=0.2,      # Сдвиг 
        scale=0.6,          # Масштаб (+-60%)
        shear=20.0,          # Сдвиг/Перекос 
        perspective=0.0001,
        
        hsv_h=0.015,        # Сдвиг оттенка. 
        hsv_s=0.3,          # Насыщенность (70%). 
        hsv_v=0.3,          # Яркость (50%). 
        bgr=0.0,            # Переворот каналов BGR->RGB 

        close_mosaic=10,
        mosaic=0.5,
        mixup=0.15,
        copy_paste=0.3,
        erasing=0.4,
       
        multi_scale=0.25
    )
    
    del rtdetr_model
    free_memory()

In [ ]:
def main():
    torch.backends.cudnn.benchmark = True          
    torch.set_float32_matmul_precision('high')     
    free_memory()
    
    for fold_idx in range(1, NUM_FOLDS + 1):
        fold_dir = os.path.join(FOLDS_DIR, f'fold_{fold_idx}')
        
        print(f"\n=========================================")
        print(f" Запуск обучения для фолда {fold_idx}")
        print(f"=========================================\n")
        
        # 1. YOLO и RT-DETR
        train_yolo11_model(fold_idx, fold_dir)

        train_rtdetr_model(fold_idx, fold_dir)
        
        # Мощная чистка при переходе к следующему фолду
        print(f"Фолд {fold_idx} завершен. Выполняется полная очистка кэша...")
        free_memory()
        
    print("Обучение всех моделей на всех фолдах завершено!")

In [18]:
if __name__ == '__main__':
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    main()


 Запуск обучения для фолда 1

--- Обучение YOLO11m на Фолде 1 ---
Ultralytics 8.4.21  Python-3.13.7 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=d:\LabsMIET\ML-Detection-object\dataset_folds\fold_1\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26m_

KeyboardInterrupt: 